<div align="center">

# 🧮 Semana 2 · Álgebra lineal: el idioma de los modelos

**Máster en Creación de Modelos de IA · Nivel I: «Construye Gleihar»** · SagaTechSchool Pro

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saga1555/SagaEstudios7316/blob/main/semana02/semana02_algebra_lineal.ipynb)

</div>

---

En el capítulo 2 viste que los datos son **vectores**, que las capas de una red son **matrices** y que todo se reduce a **multiplicar matrices**. En este cuaderno lo haces con PyTorch, la biblioteca con la que construirás Gleihar.

### 🎯 Al terminar podrás

1. Representar terrenos, imágenes y letras como **tensores** y operar con ellos.
2. Medir normas, distancias y **similitudes coseno**.
3. Multiplicar matrices, comprobar sus **formas** y medir su costo en CPU y GPU.
4. Escribir **tu primera capa** de red y recuperar $w=10{,}7$ y $b=1{,}3$ con las **ecuaciones normales**.
5. Darle a Gleihar su primera **tabla de embeddings**.

### 🗺️ Mapa del cuaderno

| Parte | Qué haces | Sección del libro | Tiempo |
|---|---|---|---|
| 0 · Tu entorno | Compruebas tu GPU | 1.7 | 5 min |
| 1 · Vectores | Operas vectores y conviertes letras a one-hot | 2.1 | 15 min |
| 2 · Parecidos | Normas, distancias y similitud coseno | 2.2 | 15 min |
| 3 · Matrices | Transformas un cuadrado con matrices | 2.3 | 15 min |
| 4 · Multiplicar y medir | $AB\neq BA$ y el costo $2n^3$ en CPU y GPU | 2.4 | 15 min |
| 5 · Tu primera capa | $XW+b$, `nn.Linear` y ecuaciones normales | 2.5 | 25 min |
| 6 · Resolver sistemas | `solve`, determinante y colinealidad | 2.6 | 15 min |
| 7 · Tensores y embeddings | Formas, GPU y la tabla de Gleihar | 2.7 | 20 min |
| 8 · Entrega | Generas tu reporte | — | 5 min |

| Símbolo | Significa |
|---|---|
| 🔮 **Predice** | Antes de ejecutar, escribe qué crees que pasará. |
| ✍️ **Tu turno** | Completa el código donde dice `...`. |
| ✅ **Comprueba** | Pregunta rápida; haz clic para ver la respuesta. |
| ⚠️ **Cuidado** | Error frecuente y cómo evitarlo. |

> 💡 Guarda tu copia con *Archivo → Guardar una copia en Drive* y activa la GPU en *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*. La Parte 4 compara CPU y GPU; todo lo demás funciona también sin GPU.

## 0️⃣ Tu entorno

In [ ]:
import sys, platform, json, time, math
import torch
import matplotlib.pyplot as plt

EN_COLAB = "google.colab" in sys.modules
GLEIHAR = {"semana": 2, "entorno": "Google Colab" if EN_COLAB else "Equipo local",
           "pytorch": torch.__version__, "dispositivo": "cuda" if torch.cuda.is_available() else "cpu"}
if torch.cuda.is_available():
    GLEIHAR["gpu"] = torch.cuda.get_device_name(0)
DISPOSITIVO = torch.device(GLEIHAR["dispositivo"])
torch.manual_seed(2026)
print(json.dumps(GLEIHAR, ensure_ascii=False, indent=2))

## 1️⃣ Vectores (sección 2.1)

En PyTorch un vector es un **tensor de un eje**. Las operaciones son componente a componente, igual que en el libro.

In [ ]:
a = torch.tensor([2., 1.])
b = torch.tensor([1., 3.])
print("a + b   =", a + b)        # (3, 4)
print("2a      =", 2 * a)        # (4, 2)
print("b - a   =", b - a)        # (-1, 2)
print("promedio=", 0.5 * a + 0.5 * b)   # combinación lineal: (1.5, 2)

### ✍️ Tu turno: letras como vectores one-hot

Completa `one_hot(texto, vocab)`: debe devolver un tensor de forma `(len(texto), len(vocab))` con un 1 en la columna de cada letra.

> 🔮 **Predice:** ¿qué da la suma de las filas de `one_hot("banana", ["a", "b", "n"])`?

In [ ]:
def one_hot(texto, vocab):
    """Ejemplo: one_hot("ba", ["a","b","n"]) → [[0,1,0],[1,0,0]]"""
    M = torch.zeros(len(texto), len(vocab))
    for i, letra in enumerate(texto):
        ...          # pon un 1 en la fila i, columna de la letra (pista: vocab.index(letra))
    return M

In [ ]:
def revisar_parte1():
    try:
        M = one_hot("banana", ["a", "b", "n"])
        conteo = M.sum(dim=0)                     # suma de las filas
        ok = M.shape == (6, 3) and torch.equal(conteo, torch.tensor([3., 1., 2.]))
        print("Conteos:", conteo.tolist(), "→ probabilidades de Gleihar v0.0:", [round(x, 3) for x in (conteo / 6).tolist()])
    except Exception as e:
        ok = False; print("Error:", e)
    print("✅ one_hot" if ok else "❌ one_hot: revisa que pongas un 1 en la columna vocab.index(letra)")
    GLEIHAR["parte1"] = "1/1" if ok else "0/1"
revisar_parte1()

> ✅ **Comprueba:** la suma de los one-hot es el vector de conteos `(3, 1, 2)`, y dividido entre 6 da las probabilidades de Gleihar v0.0. *Entrenar Gleihar v0.0 es sumar vectores y multiplicar por un escalar.*

## 2️⃣ Norma, distancia y similitud coseno (sección 2.2)

$$\lVert\mathbf v\rVert=\sqrt{\textstyle\sum v_i^2},\qquad \operatorname{sim}(\mathbf u,\mathbf v)=\frac{\mathbf u\cdot\mathbf v}{\lVert\mathbf u\rVert\,\lVert\mathbf v\rVert}$$

In [ ]:
u = torch.tensor([3., 4.])
print("norma        :", torch.linalg.norm(u).item())          # 5
print("norma L1     :", torch.linalg.norm(u, ord=1).item())   # 7
A_ = torch.tensor([2.5, 1.]); B_ = torch.tensor([1., 3.])
print("distancia A-B:", torch.linalg.norm(A_ - B_).item())     # 2.5

### ✍️ Tu turno: programa la similitud coseno usando `torch.dot` y `torch.linalg.norm`.

In [ ]:
def similitud(u, v):
    """Similitud coseno entre dos vectores (número entre -1 y 1)."""
    ...

In [ ]:
rey     = torch.tensor([0.9, 0.8, 0.1])
reina   = torch.tensor([0.85, 0.9, 0.15])
manzana = torch.tensor([0.1, 0.2, 0.95])
def revisar_parte2():
    try:
        s1, s2 = similitud(rey, reina), similitud(rey, manzana)
        ok = abs(s1 - 0.996) < 1e-3 and abs(s2 - 0.293) < 1e-3
        print(f"rey·reina: {s1:.3f}   rey·manzana: {s2:.3f}")
        ref = torch.nn.functional.cosine_similarity(rey, reina, dim=0).item()
        print(f"PyTorch también da: {ref:.3f}")
    except Exception as e:
        ok = False; print("Error:", e)
    print("✅ similitud" if ok else "❌ similitud: divide el producto punto entre el producto de las normas")
    GLEIHAR["parte2"] = "1/1" if ok else "0/1"
revisar_parte2()

## 3️⃣ Matrices como transformaciones (sección 2.3)

Aplicamos tres matrices a los vértices del cuadrado unidad. Recuerda: **las columnas de la matriz dicen a dónde van $\mathbf e_1$ y $\mathbf e_2$**.

> 🔮 **Predice:** ¿cómo se verá el cuadrado después de la cizalla $\begin{pmatrix}1&1\\0&1\end{pmatrix}$?

In [ ]:
cuadrado = torch.tensor([[0., 0.], [1., 0.], [1., 1.], [0., 1.], [0., 0.]])   # filas = puntos
matrices = {"Estirar": torch.tensor([[2., 0.], [0., 1.]]),
            "Rotar 90°": torch.tensor([[0., -1.], [1., 0.]]),
            "Cizallar": torch.tensor([[1., 1.], [0., 1.]])}
fig, ejes = plt.subplots(1, 3, figsize=(11, 3.6))
for ax, (nombre, A) in zip(ejes, matrices.items()):
    nuevo = cuadrado @ A.T          # cada punto p se transforma en A p  (como filas: p Aᵀ)
    ax.fill(cuadrado[:, 0], cuadrado[:, 1], alpha=.25, color="gray", label="original")
    ax.fill(nuevo[:, 0], nuevo[:, 1], alpha=.5, color="#5B4BD6", label="transformado")
    ax.set_title(nombre); ax.set_aspect("equal"); ax.grid(alpha=.3); ax.set_xlim(-1.5, 2.5); ax.set_ylim(-.5, 2)
ejes[0].legend(loc="upper left", fontsize=8); plt.tight_layout(); plt.show()
X = torch.tensor([[2.5, 1.], [1., 3.], [2.2, 1.5]])
print("Forma de X:", tuple(X.shape), "  Forma de Xᵀ:", tuple(X.T.shape))

> ⚠️ **Cuidado:** como guardamos los puntos como **filas**, aplicar $A$ a cada punto se escribe `puntos @ A.T`. Es la misma regla $(AB)^\mathsf{T}=B^\mathsf{T}A^\mathsf{T}$ de la sección 2.4.

## 4️⃣ Multiplicar matrices y medir el costo (sección 2.4)

In [ ]:
A = torch.tensor([[1., 2.], [3., 4.]])
B = torch.tensor([[0., 1.], [1., 0.]])
print("AB =\n", A @ B)
print("BA =\n", B @ A)
print("¿AB == BA?", torch.equal(A @ B, B @ A))
print("¿(AB)ᵀ == BᵀAᵀ?", torch.equal((A @ B).T, B.T @ A.T))

Ahora medimos el tiempo de multiplicar matrices $n\times n$. Según la sección 2.4, cuesta unas $2n^3$ operaciones.

> 🔮 **Predice:** si $n$ se multiplica por 4, ¿por cuánto se multiplica el tiempo?

In [ ]:
def medir(n, dispositivo, repeticiones=3):
    A = torch.randn(n, n, device=dispositivo); B = torch.randn(n, n, device=dispositivo)
    A @ B                                   # calentamiento
    if dispositivo.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(repeticiones): A @ B
    if dispositivo.type == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t0) / repeticiones

resultados = {}
for n in [500, 1000, 2000]:
    t_cpu = medir(n, torch.device("cpu"))
    fila = {"cpu_s": round(t_cpu, 4), "gflops_cpu": round(2 * n**3 / t_cpu / 1e9, 1)}
    if torch.cuda.is_available():
        t_gpu = medir(n, torch.device("cuda"))
        fila.update(gpu_s=round(t_gpu, 5), gflops_gpu=round(2 * n**3 / t_gpu / 1e9, 1))
    resultados[n] = fila
    print(n, fila)
GLEIHAR["parte4"] = {str(k): v for k, v in resultados.items()}

> ✅ **Comprueba:** al pasar de $n=500$ a $n=2000$ (4 veces más), el tiempo crece unas $4^3=64$ veces. En la GPU, la columna `gflops_gpu` suele ser decenas de veces mayor que la de la CPU: por eso entrenamos en GPU.

## 5️⃣ Tu primera capa (sección 2.5)

$$Z = XW + \mathbf b,\qquad (n\times d)(d\times k)=(n\times k)$$

### ✍️ Tu turno: calcula la capa del ejemplo resuelto del libro para los dos terrenos.

In [ ]:
X = torch.tensor([[2.5, 1.], [1., 3.]])            # 2 terrenos: superficie, distancia
W = torch.tensor([[2., 1., 0.], [-1., 0., 1.]])    # 3 neuronas (columnas)
b = torch.tensor([-1., 0., 0.5])
Z = ...                                            # usa el operador @ y suma b
A = torch.relu(Z)

In [ ]:
def revisar_parte5a():
    try:
        ok = torch.allclose(Z, torch.tensor([[3., 2.5, 1.5], [-2., 1., 3.5]]))
        print("Z =\n", Z, "\nA = ReLU(Z) =\n", A)
        capa = torch.nn.Linear(2, 3)                 # PyTorch guarda W transpuesta: forma (3, 2)
        with torch.no_grad():
            capa.weight.copy_(W.T); capa.bias.copy_(b)
        ok = ok and torch.allclose(capa(X), Z)
        print("¿nn.Linear da lo mismo?", torch.allclose(capa(X), Z))
    except Exception as e:
        ok = False; print("Error:", e)
    print("✅ capa" if ok else "❌ capa: Z debe ser X @ W + b")
    GLEIHAR["parte5_capa"] = "1/1" if ok else "0/1"
revisar_parte5a()

### Las ecuaciones normales recuperan la recta de los terrenos

$$X^\mathsf{T}X\,\boldsymbol\theta = X^\mathsf{T}\mathbf y$$

> 🔮 **Predice:** ¿qué valores de $w$ y $b$ deberían salir?

In [ ]:
superficie = torch.tensor([1.0, 1.5, 2.0, 2.5, 3.0], dtype=torch.float64)
precio     = torch.tensor([11.5, 18.0, 22.5, 28.5, 33.0], dtype=torch.float64)
Xd = torch.stack([superficie, torch.ones(5, dtype=torch.float64)], dim=1)   # columna de unos para b
y = precio
print("XᵀX =\n", Xd.T @ Xd)
print("Xᵀy =", Xd.T @ y)
theta = torch.linalg.solve(Xd.T @ Xd, Xd.T @ y)
print(f"w = {theta[0]:.3f}   b = {theta[1]:.3f}")
residuos = y - Xd @ theta
print("Xᵀ · residuos =", (Xd.T @ residuos).round(decimals=10), "← cero: los residuos son ortogonales a las columnas")
GLEIHAR["parte5_normales"] = {"w": round(theta[0].item(), 3), "b": round(theta[1].item(), 3)}

## 6️⃣ Resolver sistemas (sección 2.6)

In [ ]:
M = torch.tensor([[1., 1., 1.], [0., 2., 5.], [2., 5., -1.]])
r = torch.tensor([6., -4., 27.])
print("Solución del sistema 3×3:", torch.linalg.solve(M, r))          # (5, 3, -2)
P = torch.tensor([[2., 3.], [1., -1.]])
print("Banano y naranja:", torch.linalg.solve(P, torch.tensor([13., -1.])), "  det =", torch.linalg.det(P).item())

# Colinealidad: agregamos la superficie en m² (proporcional a la de cientos de m²)
Xc = torch.stack([superficie, 100 * superficie, torch.ones(5, dtype=torch.float64)], dim=1)
print("det(XᵀX) con una columna proporcional:", torch.linalg.det(Xc.T @ Xc).item())

> ✅ **Comprueba:** el determinante sale 0 o un número diminuto por redondeo. Las ecuaciones normales ya no tienen solución única: la información de la columna nueva estaba repetida.

## 7️⃣ Tensores, GPU y la tabla de embeddings de Gleihar (sección 2.7)

In [ ]:
lote = torch.randn(32, 3, 224, 224)
print("forma:", tuple(lote.shape), " elementos:", lote.numel(), f" memoria: {lote.numel()*4/1e6:.1f} MB")
plano = lote.reshape(32, -1)
print("aplanado:", tuple(plano.shape))
print("difusión (4,3)+(3,):", tuple((torch.ones(4, 3) + torch.ones(3)).shape))
if torch.cuda.is_available():
    print("en GPU:", plano.to("cuda").device)

Ahora Gleihar recibe su **tabla de embeddings**: un vector de 16 números para cada símbolo del Quijote. Por ahora los vectores son **aleatorios**; en la semana 6 se entrenarán y las letras que se usan parecido quedarán cerca.

In [ ]:
texto = """En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha mucho tiempo que vivía un hidalgo
de los de lanza en astillero, adarga antigua, rocín flaco y galgo corredor. Una olla de algo más vaca que carnero,
salpicón las más noches, duelos y quebrantos los sábados, lantejas los viernes, algún palomino de añadidura los
domingos, consumían las tres partes de su hacienda. El resto della concluían sayo de velarte, calzas de velludo para
las fiestas, con sus pantuflos de lo mesmo, y los días de entresemana se honraba con su vellorí de lo más fino."""
texto = texto.lower().replace("\n", " ")      # el mismo texto de la semana 1
letras = sorted(set(texto))
torch.manual_seed(2026)
E = torch.nn.Embedding(len(letras), 16)
print(f"Vocabulario: {len(letras)} símbolos · tabla E de forma {tuple(E.weight.shape)} · parámetros: {E.weight.numel()}")

indices = torch.tensor([letras.index(c) for c in "banana" if c in letras])
print("«banana» como embeddings:", tuple(E(indices).shape))

# Un embedding es one-hot por matriz: comprobamos con la letra «a»
oh = torch.zeros(len(letras)); oh[letras.index("a")] = 1
print("¿one-hot @ E == fila de E?", torch.allclose(oh @ E.weight, E.weight[letras.index("a")]))

# ¿Qué letras se parecen ANTES de entrenar? (al azar)
V = torch.nn.functional.normalize(E.weight.detach(), dim=1)
S = V @ V.T                                            # matriz de similitudes coseno
S.fill_diagonal_(-1)
i, j = divmod(S.argmax().item(), len(letras))
print(f"Par más parecido antes de entrenar: «{letras[i]}» y «{letras[j]}» (sim = {S.max():.2f}) — puro azar")
GLEIHAR["parte7"] = {"vocabulario": len(letras), "forma_E": list(E.weight.shape), "par_azar": [letras[i], letras[j]]}

## 8️⃣ Tu reporte

Copia el resultado y pégalo en el cuadro de texto de la **Tarea 2**.

In [ ]:
GLEIHAR["fecha"] = time.strftime("%Y-%m-%d %H:%M")
with open("reporte_semana02.json", "w", encoding="utf-8") as f:
    json.dump(GLEIHAR, f, ensure_ascii=False, indent=2)
print(json.dumps(GLEIHAR, ensure_ascii=False, indent=2))
print("\n✅ Copia este reporte y pégalo en la Tarea 2.")

---

## 🏁 Lo que aprendiste hoy

- ✔️ Los datos son **vectores**, y entrenar Gleihar v0.0 era sumar vectores one-hot.
- ✔️ La **similitud coseno** mide parecidos sin importar el tamaño.
- ✔️ Una **capa** es $XW+\mathbf b$, y `nn.Linear` hace exactamente eso.
- ✔️ Las **ecuaciones normales** recuperan $w=10{,}7$ y $b=1{,}3$.
- ✔️ Gleihar ya tiene su **tabla de embeddings** (por ahora al azar).

### 🔜 La próxima semana

**Derivadas, gradiente y probabilidad:** de dónde sale $\nabla L$ y cómo PyTorch lo calcula solo con `backward()`.